In [1]:
import pandas as pd

X_train = pd.read_csv("../data/X_train_tree.csv")

X_test = pd.read_csv(
    "../data/X_test_tree.csv"
)

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")

In [2]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [3]:
from xgboost import XGBRegressor

from src.evaluate import rmse_cv

In [4]:
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb_rmse = rmse_cv(
    xgb_model,
    X_train,
    y_train
).mean()

print(
    f"XGBoost RMSE: {xgb_rmse:.5f}"
)

XGBoost RMSE: 0.11796


In [5]:
X_test = pd.read_csv("../data/X_test_tree.csv")
test = pd.read_csv("../data/test.csv")

print(X_test.shape)
print(test.shape)

(1459, 303)
(1459, 80)


In [6]:
xgb_model.fit(
    X_train,
    y_train
)

print("XGB 訓練完成")

XGB 訓練完成


In [7]:
import numpy as np

# 1. 使用訓練好的 xgb_model 對測試集進行預測
xgb_preds_log = xgb_model.predict(X_test)

# 2. 將對數預測結果還原為真實房價 (非常重要！必須使用 np.expm1)
xgb_preds = np.expm1(xgb_preds_log)

# 3. 建立符合 Kaggle 格式的 DataFrame
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_preds
})

# 顯示前五筆資料，確認房價是否合理 (應該要落在十幾萬上下)
submission.head()

,Id,SalePrice
0,1461,122719.500000
1,1462,165269.515625
2,1463,180088.578125
3,1464,190619.906250
4,1465,177440.328125


In [8]:
submission.to_csv(
    "../submissions/xgb_submission.csv",
    index=False
)

print("XGB Submission 已建立")

XGB Submission 已建立


In [9]:
# XGBoost v2

xgb_model_v2 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v2 = rmse_cv(
    xgb_model_v2,
    X_train,
    y_train
).mean()

print(
    f"XGB v2 RMSE: {xgb_rmse_v2:.5f}"
)

XGB v2 RMSE: 0.11221


In [10]:
xgb_model_v2.fit(
    X_train,
    y_train
)

print("XGB v2 訓練完成")

XGB v2 訓練完成


In [11]:
import numpy as np
import pandas as pd

# 1. 使用訓練好的 xgb_model_v2 對測試集進行預測
xgb_preds_log_v2 = xgb_model_v2.predict(X_test)

# 2. 將對數預測結果還原為真實房價 (專案規範：必須使用 np.expm1 還原)
xgb_preds_v2 = np.expm1(xgb_preds_log_v2)

# 3. 建立符合 Kaggle 格式的 DataFrame，並命名為 submission_v2
submission_v2 = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_preds_v2
})

# 顯示前五筆資料，確認房價是否落在合理範圍 (約十幾萬上下)
submission_v2.head()

,Id,SalePrice
0,1461,123499.437500
1,1462,160863.625000
2,1463,178421.296875
3,1464,192394.937500
4,1465,184274.906250


In [12]:
submission_v2.to_csv(
    "../submissions/xgb_v2_submission.csv",
    index=False
)

print("XGB v2 Submission 已建立")

XGB v2 Submission 已建立


In [13]:
# XGBoost v3

xgb_model_v3 = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.01,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v3 = rmse_cv(
    xgb_model_v3,
    X_train,
    y_train
).mean()

print(
    f"XGB v3 RMSE: {xgb_rmse_v3:.5f}"
)

XGB v3 RMSE: 0.11394


In [14]:
xgb_model_v4 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)
xgb_rmse_v4 = rmse_cv(
    xgb_model_v4,
    X_train,
    y_train
).mean()

print(
    f"XGB v4 RMSE: {xgb_rmse_v4:.5f}"
)

XGB v4 RMSE: 0.11256


In [15]:
import numpy as np
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

param_grid = {
    "n_estimators": np.arange(1500, 3500, 200),
    "learning_rate": [0.01, 0.015, 0.02, 0.025],
    "max_depth": [3, 4],
    "subsample": [0.75, 0.8, 0.85],
    "colsample_bytree": [0.75, 0.8, 0.85],
    "min_child_weight": [1, 2, 3],
}

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,   # 先用10
    scoring="neg_root_mean_squared_error",
    cv=kf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("開始搜索...")

random_search.fit(
    X_train,
    y_train
)

print("\n=== 搜索完成 ===")
print(random_search.best_params_)
print(
    f"RMSE: {-random_search.best_score_:.5f}"
)

開始搜索...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

=== 搜索完成 ===
{'subsample': 0.75, 'n_estimators': np.int64(2500), 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.02, 'colsample_bytree': 0.8}
RMSE: 0.11271


In [16]:
xgb_best = XGBRegressor(
    n_estimators=3500,
    learning_rate=0.015,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)

xgb_best.fit(
    X_train,
    y_train
)

print("最佳 XGB 訓練完成")

最佳 XGB 訓練完成


In [17]:
xgb_best_pred_log = xgb_best.predict(
    X_test
)

xgb_best_pred = np.expm1(
    xgb_best_pred_log
)

In [18]:
submission_best = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_best_pred
})

submission_best.head()

,Id,SalePrice
0,1461,124413.828125
1,1462,161151.531250
2,1463,180115.203125
3,1464,193455.812500
4,1465,183781.906250


In [19]:
submission_best.to_csv(
    "../submissions/xgb_random_search_submission.csv",
    index=False
)

print("XGB Random Search Submission 已建立")

XGB Random Search Submission 已建立


In [20]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

model = XGBRegressor(
    n_estimators=3500,
    learning_rate=0.015,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse = np.sqrt(
    -cross_val_score(
        model,
        X_train,
        y_train,
        scoring="neg_mean_squared_error",
        cv=kf
    )
)

print(rmse.mean())

0.11276353245414476


In [21]:
print(X_train.shape)
print(X_test.shape)

(1458, 303)
(1459, 303)
